# Bronings University Hospital Group: Weekly Waiting List Exception Report

**Purpose:** automate the Monday-morning exception report that Sarah Okafor currently builds by hand in about 90 minutes.
**Owner for handover:** Daniel Walsh (Data Engineering)
**Run time:** under 60 seconds
**Output:** `output/Waiting_List_Exception_Report_<date>.xlsx`

The notebook follows five steps, and every rule matches the Phase 3 DQ Issue Log and the Phase 5 KPI Framework:

| Step | What it does |
|---|---|
| 1. Load | Read the PAS outpatient extract (and the four other datasets for DQ scoring) with relative paths |
| 2. Clean | Remove non-standard attendance statuses, flag blank NHS Numbers, recode the retired GEN code, validate Waiting_Days |
| 3. Analyse | DNA rate by specialty (KPI 03), RTT 18-week breaches on open pathways (KPI 04/06), waiting-time statistics |
| 4. Score | A Data Quality Score (completeness, validity, uniqueness) for each of the five datasets |
| 5. Export | A formatted Excel report, ready for the Operations Director with no reformatting |

**Folder layout (relative paths, so the notebook is portable):**
```
Bronings_Automation/
├── Bronings_Waiting_List_Exception_Report.ipynb
├── data/      03_outpatient_appointments.csv, 04_inpatient_admissions.csv, 05_ed_attendances.csv,
│              06_theatre_utilisation.csv, 07_workforce_staffing.csv
└── output/    (created automatically)
```

## 0. Setup: libraries, paths and business rules
All thresholds live in one place, so a rule change (for example a new specialty code mapping) is a one-line edit, not a hunt through the code.

In [1]:
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

RUN_START = time.time()
RUN_DATE = datetime.now()

DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

FILES = {
    "Outpatient":  DATA_DIR / "03_outpatient_appointments.csv",
    "Inpatient":   DATA_DIR / "04_inpatient_admissions.csv",
    "ED":          DATA_DIR / "05_ed_attendances.csv",
    "Theatre":     DATA_DIR / "06_theatre_utilisation.csv",
    "Workforce":   DATA_DIR / "07_workforce_staffing.csv",
}

# Business rules (Phase 5 KPI Framework)
VALID_STATUS      = ["ATT", "DNA", "CAN", "REF"]   # national attendance codes
RTT_LIMIT_DAYS    = 126                           # 18 weeks
LONG_WAIT_DAYS    = 364                           # 52 weeks
VERY_LONG_DAYS    = 728                           # 104 weeks
DNA_TARGET_PCT    = 8.0                           # Trust target
RTT_STANDARD_PCT  = 92.0                          # NHS constitutional standard
SPECIALTY_MAP     = {"GEN": "GM"}                 # retired code -> current national code
VALID_CLINICS     = [f"CL-1{n:02d}" for n in range(1, 11)]   # CL-101 to CL-110
WTR_MAX_HOURS     = 13.0                          # Working Time Regulations shift limit

run_log = []   # audit trail written to the Excel report
def log(step, detail):
    run_log.append({"Step": step, "Detail": detail, "Time": datetime.now().strftime("%H:%M:%S")})
    print(f"[{step}] {detail}")

## 1. Load
Every column is read as **text** (`dtype=str`). This matters for NHS data: a 10-digit NHS Number such as `0506298481` loses its leading zero if pandas reads it as a number (Pipeline Issue 1). Blanks stay as empty strings (`keep_default_na=False`), so "blank" can be counted precisely instead of mixing with genuine NaNs.

In [2]:
def load(name):
    df = pd.read_csv(FILES[name], dtype=str, keep_default_na=False)
    df = df.apply(lambda s: s.str.strip())          # remove stray spaces
    log("Load", f"{name}: {len(df):,} rows x {df.shape[1]} columns")
    return df

raw = {name: load(name) for name in FILES}
op_raw = raw["Outpatient"]
op_raw.head()

[Load] Outpatient: 580 rows x 15 columns
[Load] Inpatient: 1,206 rows x 17 columns
[Load] ED: 969 rows x 14 columns
[Load] Theatre: 208 rows x 11 columns
[Load] Workforce: 1,560 rows x 12 columns


,Appointment_ID,NHS_Number,Patient_Surname,Patient_DOB,Specialty_Code,Consultant_Code,Clinic_Code,Appointment_Date,Appointment_Time,Attendance_Status,Waiting_Days,RTT_Status,RTT_Weeks,Referral_Source,Postcode
0,APT-100001,4823719056,AHMED,15/03/1978,GM,C-1042,CL-101,06/01/2025,09:00,ATT,34,CLOSED,4.9,GP,NG1 2AB
1,APT-100002,7612384905,PATEL,22/07/1965,CS,C-1017,CL-102,06/01/2025,09:30,ATT,67,CLOSED,9.6,GP,LE1 1AA
2,APT-100003,3394820174,WILSON,08/11/1989,OR,C-1033,CL-103,06/01/2025,10:00,DNA,145,OPEN,20.7,GP,NG5 3CD
3,APT-100004,9182746350,OKAFOR,31/01/1972,GS,C-1028,CL-104,06/01/2025,10:30,ATT,28,CLOSED,4.0,CONS,LE2 7GH
4,APT-100005,6547832901,KHAN,14/06/1956,RS,C-1051,CL-105,06/01/2025,11:00,ATT,89,CLOSED,12.7,GP,NG7 4EF


## 2. Clean the outpatient appointments
Each rule below fixes an issue documented in the DQ Issue Log. Rows are **flagged rather than silently deleted** wherever possible, so the exception report can show what was excluded and why. That is the audit trail the manual process lacked.

| Rule | Issue log ref | Action |
|---|---|---|
| Non-standard `Attendance_Status` (`attended`, `N/A`) | DQ-001 | Excluded from rate calculations, listed as exceptions |
| Blank `NHS_Number` | DQ-002 | Kept for activity counts, flagged as exceptions (cannot be patient-linked) |
| Retired specialty code `GEN` | DQ-021 | Recoded to `GM` |
| `Waiting_Days` stored as text (`ATT`) | DQ-019 | Derived from `RTT_Weeks x 7` and flagged |
| Negative `Waiting_Days` | DQ-018 | Excluded from waiting-time figures |
| Invalid clinic codes (`CL-OLD`, `CL-DEP`) | DQ-024 | Flagged as exceptions |

In [3]:
op = op_raw.copy()

# 2a. Specialty recode (GEN -> GM)
gen_before = (op["Specialty_Code"] == "GEN").sum()
op["Specialty_Code"] = op["Specialty_Code"].replace(SPECIALTY_MAP)
log("Clean", f"Specialty GEN recoded to GM: {gen_before} rows")

# 2b. Attendance status: case-sensitive check against the national code list
op["Status_Valid"] = op["Attendance_Status"].isin(VALID_STATUS)
nonstandard = op.loc[~op["Status_Valid"], ["Appointment_ID", "Attendance_Status"]]
log("Clean", f"Non-standard Attendance_Status excluded: {len(nonstandard)} -> {nonstandard['Attendance_Status'].tolist()}")

# 2c. Blank NHS Numbers: flag, do not drop
op["NHS_Blank"] = op["NHS_Number"].eq("")
log("Clean", f"Blank NHS_Number flagged: {op['NHS_Blank'].sum()}")

# 2d. Waiting_Days validation
wd  = pd.to_numeric(op["Waiting_Days"], errors="coerce")   # 'ATT' -> NaN
rtw = pd.to_numeric(op["RTT_Weeks"], errors="coerce")
op["Waiting_Days_Flag"] = np.select(
    [wd.isna() & rtw.notna(), wd < 0, wd.notna()],
    ["DERIVED from RTT_Weeks x7", "NEGATIVE - excluded", "OK"],
    default="NO VALUE")
op["Waiting_Days_Valid"] = np.where(wd.isna(), np.round(rtw * 7), wd)
op.loc[wd < 0, "Waiting_Days_Valid"] = np.nan
log("Clean", f"Waiting_Days: {op['Waiting_Days_Flag'].value_counts().to_dict()}")

# 2e. Invalid clinic codes
op["Clinic_Valid"] = op["Clinic_Code"].isin(VALID_CLINICS)
log("Clean", f"Invalid clinic codes flagged: {(~op['Clinic_Valid']).sum()} -> {op.loc[~op['Clinic_Valid'], 'Clinic_Code'].tolist()}")

op_clean = op[op["Status_Valid"]].copy()     # analysis set
log("Clean", f"Analysis set: {len(op_clean)} of {len(op)} appointments")

[Clean] Specialty GEN recoded to GM: 3 rows
[Clean] Non-standard Attendance_Status excluded: 2 -> ['attended', 'N/A']
[Clean] Blank NHS_Number flagged: 3
[Clean] Waiting_Days: {'OK': 551, 'DERIVED from RTT_Weeks x7': 28, 'NEGATIVE - excluded': 1}
[Clean] Invalid clinic codes flagged: 2 -> ['CL-OLD', 'CL-DEP']
[Clean] Analysis set: 578 of 580 appointments


## 3a. Analyse: DNA rate by specialty (KPI 03)
**Formula (signed off at Phase 5):** `DNA / (ATT + DNA) x 100`.
CAN (hospital cancellation) and REF (patient refusal) are excluded from the denominator, because the patient never had the chance to miss a slot that was cancelled. Including them, as the old 9.8% formula did, dilutes the rate.

In [4]:
att_dna = op_clean[op_clean["Attendance_Status"].isin(["ATT", "DNA"])]

dna = (att_dna.groupby("Specialty_Code")["Attendance_Status"]
              .agg(Attended=lambda s: (s == "ATT").sum(),
                   DNA=lambda s: (s == "DNA").sum())
              .reset_index())
dna["Denominator"] = dna["Attended"] + dna["DNA"]
dna["DNA_Rate_Pct"] = (dna["DNA"] / dna["Denominator"] * 100).round(2)
dna["vs_8pct_Target"] = np.where(dna["DNA_Rate_Pct"] < DNA_TARGET_PCT, "Below target", "ABOVE TARGET")
dna = dna.sort_values("DNA_Rate_Pct", ascending=False).reset_index(drop=True)

trust_dna = round(dna["DNA"].sum() / dna["Denominator"].sum() * 100, 2)
log("Analyse", f"Trust DNA rate {trust_dna}% ({dna['DNA'].sum()} / {dna['Denominator'].sum()})")
dna

[Analyse] Trust DNA rate 14.1% (76 / 539)


,Specialty_Code,Attended,DNA,Denominator,DNA_Rate_Pct,vs_8pct_Target
0,OR,27,17,44,38.64,ABOVE TARGET
1,URO,42,11,53,20.75,ABOVE TARGET
2,OPH,48,10,58,17.24,ABOVE TARGET
3,CS,41,8,49,16.33,ABOVE TARGET
4,RS,50,9,59,15.25,ABOVE TARGET
5,GYN,51,8,59,13.56,ABOVE TARGET
6,RHE,59,5,64,7.81,Below target
7,GS,49,4,53,7.55,Below target
8,GM,52,4,56,7.14,Below target
9,EM,44,0,44,0.00,Below target


## 3b. Analyse: RTT 18-week breaches (KPI 04 and KPI 06)
NHS England measures RTT on **open (incomplete) pathways**. A breach is an open pathway with a validated wait over 126 days. The patient-level list is the core of the exception report: it is what the specialty PTL meetings work from.

In [5]:
open_pw = op[(op["RTT_Status"] == "OPEN") & op["Waiting_Days_Valid"].notna()].copy()
open_pw["Weeks_Waiting"] = (open_pw["Waiting_Days_Valid"] / 7).round(1)
open_pw["Breach_18wk"] = open_pw["Waiting_Days_Valid"] > RTT_LIMIT_DAYS
open_pw["Wait_Band"] = pd.cut(open_pw["Waiting_Days_Valid"],
                              bins=[-1, RTT_LIMIT_DAYS, LONG_WAIT_DAYS, VERY_LONG_DAYS, np.inf],
                              labels=["0-18 wks", "18-52 wks", "52-104 wks", ">104 wks"])

rtt_breaches = open_pw["Breach_18wk"].sum()
rtt_total = len(open_pw)
rtt_compliance = round((1 - rtt_breaches / rtt_total) * 100, 2)
over_52 = (open_pw["Waiting_Days_Valid"] > LONG_WAIT_DAYS).sum()
over_104 = (open_pw["Waiting_Days_Valid"] > VERY_LONG_DAYS).sum()
log("Analyse", f"RTT breaches {rtt_breaches} of {rtt_total} open pathways; compliance {rtt_compliance}%; >52wks {over_52}; >104wks {over_104}")

rtt_spec = (open_pw.groupby("Specialty_Code")
                   .agg(Open_Pathways=("Appointment_ID", "count"),
                        Breaching_18wk=("Breach_18wk", "sum"))
                   .reset_index())
rtt_spec["Compliance_Pct"] = ((1 - rtt_spec["Breaching_18wk"] / rtt_spec["Open_Pathways"]) * 100).round(2)
rtt_spec["RAG"] = np.select([rtt_spec["Compliance_Pct"] >= 92, rtt_spec["Compliance_Pct"] >= 85], ["Green", "Amber"], "Red")
rtt_spec = rtt_spec.sort_values("Compliance_Pct").reset_index(drop=True)

breach_list = (open_pw[open_pw["Breach_18wk"]]
               .sort_values("Waiting_Days_Valid", ascending=False)
               [["Appointment_ID", "NHS_Number", "Specialty_Code", "Consultant_Code", "Clinic_Code",
                 "Appointment_Date", "Waiting_Days_Valid", "Weeks_Waiting", "Wait_Band", "Waiting_Days_Flag"]]
               .rename(columns={"Waiting_Days_Valid": "Waiting_Days"}))
rtt_spec

[Analyse] RTT breaches 76 of 130 open pathways; compliance 41.54%; >52wks 2; >104wks 0


,Specialty_Code,Open_Pathways,Breaching_18wk,Compliance_Pct,RAG
0,RS,9,9,0.00,Red
1,OR,19,17,10.53,Red
2,GYN,9,8,11.11,Red
3,RHE,6,5,16.67,Red
4,OPH,12,10,16.67,Red
5,GS,5,4,20.00,Red
6,CS,10,8,20.00,Red
7,URO,16,11,31.25,Red
8,GM,7,4,42.86,Red
9,EM,37,0,100.00,Green


## 3c. Analyse: waiting-time statistics (open pathways)
Average, median and 90th percentile by specialty. The 90th percentile uses the same method as Excel `PERCENTILE.INC`, so the figures reconcile with the Excel workbook.

In [6]:
wait_stats = (open_pw.groupby("Specialty_Code")["Waiting_Days_Valid"]
                     .agg(Open_Pathways="count", Mean_Days="mean", Median_Days="median",
                          P90_Days=lambda s: s.quantile(0.9), Max_Days="max")
                     .round(1).reset_index()
                     .sort_values("Mean_Days", ascending=False).reset_index(drop=True))
trust_wait = {
    "Mean_Days": round(open_pw["Waiting_Days_Valid"].mean(), 2),
    "Median_Days": open_pw["Waiting_Days_Valid"].median(),
    "P90_Days": open_pw["Waiting_Days_Valid"].quantile(0.9),
    "Max_Days": open_pw["Waiting_Days_Valid"].max(),
}
log("Analyse", f"Open-pathway waits: {trust_wait}")
wait_stats

[Analyse] Open-pathway waits: {'Mean_Days': np.float64(138.54), 'Median_Days': np.float64(156.0), 'P90_Days': np.float64(267.0), 'Max_Days': np.float64(423.0)}


,Specialty_Code,Open_Pathways,Mean_Days,Median_Days,P90_Days,Max_Days
0,OR,19,253.2,268.0,360.4,423.0
1,RHE,6,190.0,184.5,278.5,355.0
2,OPH,12,179.2,184.0,242.9,246.0
3,GM,7,172.6,245.0,285.0,312.0
4,RS,9,171.0,168.0,190.0,190.0
5,URO,16,161.6,183.5,229.0,235.0
6,GS,5,158.0,178.0,182.0,184.0
7,CS,10,156.1,189.5,202.1,203.0
8,GYN,9,151.1,144.0,180.4,230.0
9,EM,37,23.4,18.0,32.8,99.0


## 4. Score: Data Quality Score for each dataset
One reusable function scores any dataset on three DAMA/NHS DQMI-style dimensions:

* **Completeness:** % of required fields that are populated
* **Validity:** % of rows passing every business rule for that dataset
* **Uniqueness:** % of rows whose record ID is not duplicated

**DQ Score = average of the three, out of 100.** RAG: Green 95 or above, Amber 85 to 94.9, Red below 85.
The rules are taken directly from the Phase 3 DQ Issue Log, so the score moves when those issues are fixed at source. That makes it a usable weekly figure for the DSP Toolkit.

In [7]:
def dq_score(df, name, id_col, required, rules):
    """Return a one-row summary and a long table of failing rules for one dataset.

    df        raw dataframe (all text)
    id_col    record identifier for the uniqueness check (None = skip)
    required  columns that must be populated
    rules     dict {rule description: boolean Series that is True when the row PASSES}
    """
    n = len(df)
    cells = n * len(required)
    blanks = sum((df[c] == "").sum() for c in required)
    completeness = (1 - blanks / cells) * 100

    passes = pd.concat(rules.values(), axis=1).all(axis=1) if rules else pd.Series(True, index=df.index)
    validity = passes.mean() * 100

    if id_col:
        dup_rows = df.duplicated(id_col, keep="first").sum()
        uniqueness = (1 - dup_rows / n) * 100
    else:
        dup_rows, uniqueness = 0, 100.0

    score = round((completeness + validity + uniqueness) / 3, 1)
    rag = "Green" if score >= 95 else "Amber" if score >= 85 else "Red"
    summary = {"Dataset": name, "Rows": n, "Completeness_Pct": round(completeness, 2),
               "Validity_Pct": round(validity, 2), "Uniqueness_Pct": round(uniqueness, 2),
               "Blank_Required_Cells": int(blanks), "Rows_Failing_Rules": int((~passes).sum()),
               "Duplicate_IDs": int(dup_rows), "DQ_Score": score, "RAG": rag}
    failures = [{"Dataset": name, "Rule": r, "Rows_Failing": int((~ok).sum())} for r, ok in rules.items()]
    return summary, failures

num = lambda s: pd.to_numeric(s, errors="coerce")
date_ok = lambda s: pd.to_datetime(s, format="%d/%m/%Y", errors="coerce").notna() | (s == "")

In [8]:
o, i, e, t, w = (raw[k] for k in ["Outpatient", "Inpatient", "ED", "Theatre", "Workforce"])

specs = [
  ("Outpatient (PAS)", o, "Appointment_ID",
   ["NHS_Number", "Specialty_Code", "Consultant_Code", "Clinic_Code", "Appointment_Date", "Attendance_Status", "RTT_Status", "RTT_Weeks"],
   {"Attendance_Status in ATT/DNA/CAN/REF": o["Attendance_Status"].isin(VALID_STATUS),
    "Waiting_Days numeric and >= 0":        num(o["Waiting_Days"]) >= 0,
    "Specialty_Code not retired (GEN)":     o["Specialty_Code"] != "GEN",
    "Clinic_Code active":                   o["Clinic_Code"].isin(VALID_CLINICS),
    "Appointment_Time is HH:MM":            o["Appointment_Time"].str.fullmatch(r"\d{2}:\d{2}")}),
  ("Inpatient (EHR)", i, "Episode_ID",
   ["NHS_Number", "Admission_Date", "Ward_Code", "Specialty_Code", "HRG_Code", "Consultant_Code", "ICD10_Code", "OPCS4_Code", "Readmission_Flag"],
   {"Admission_Type in ELEC/EMER/TRANSFER": i["Admission_Type"].isin(["ELEC", "EMER", "TRANSFER"]),
    "Discharge_Date DD/MM/YYYY or blank":   date_ok(i["Discharge_Date"]),
    "Length_of_Stay >= 0 or blank":         (num(i["Length_of_Stay"]) >= 0) | (i["Length_of_Stay"] == ""),
    "Ward not decommissioned (W-08)":       i["Ward_Code"] != "W-08",
    "Specialty_Code not retired (GEN)":     i["Specialty_Code"] != "GEN"}),
  ("ED (EDIS)", e, "Attendance_ID",
   ["NHS_Number", "Arrival_Date", "Arrival_Time", "Triage_Category", "Departure_Time", "Four_Hour_Breach"],
   {"Triage_Category 1-5":                  num(e["Triage_Category"]).between(1, 5),
    "Arrival_Time is 24h HH:MM":            e["Arrival_Time"].str.fullmatch(r"\d{1,2}:\d{2}"),
    "Total_ED_Time_Minutes >= 0":           num(e["Total_ED_Time_Minutes"]) >= 0,
    "Departure_Code defined (not LEFT)":    e["Departure_Code"] != "LEFT"}),
  ("Theatre (TMS)", t, None,
   ["Week_Ending", "Theatre_ID", "Sessions_Available", "Sessions_Used", "Cancellations_Trust", "Cancellations_Patient"],
   {"Theatre_ID in T1-T4":                  t["Theatre_ID"].isin(["T1", "T2", "T3", "T4"]),
    "Sessions_Used <= Sessions_Available":  num(t["Sessions_Used"]) <= num(t["Sessions_Available"]),
    "Utilisation_Pct = Used/Available":     ((num(t["Sessions_Used"]) / num(t["Sessions_Available"]).replace(0, np.nan) * 100).round(1).fillna(0)
                                             - num(t["Utilisation_Pct"])).abs() <= 0.05,
    "Week_Ending is a Sunday":              pd.to_datetime(t["Week_Ending"], format="%d/%m/%Y").dt.dayofweek == 6}),
  ("Workforce (WRS)", w, "Shift_ID",
   ["Staff_ID", "Ward_Code", "Shift_Date", "Agency_Flag", "Absence_Flag"],
   {"Not an ST-99 test record":             ~w["Staff_ID"].str.startswith("ST-99"),
    "Shift_Type in DAY/NIGHT/LONG":         w["Shift_Type"].isin(["DAY", "NIGHT", "LONG"]),
    "Overtime_Hours >= 0":                  num(w["Overtime_Hours"]) >= 0,
    f"Hours_Worked <= {WTR_MAX_HOURS:g}h (WTR)": num(w["Hours_Worked"]) <= WTR_MAX_HOURS,
    "Ward not decommissioned (W-08)":       w["Ward_Code"] != "W-08"}),
]

summaries, failures = [], []
for name, df, idc, req, rules in specs:
    s, f = dq_score(df, name, idc, req, rules)
    summaries.append(s); failures += f
dq_summary = pd.DataFrame(summaries)
dq_rules = pd.DataFrame(failures)
trust_dq = round(dq_summary["DQ_Score"].mean(), 1)
log("Score", f"Trust DQ Score {trust_dq} | " + ", ".join(f"{r.Dataset.split()[0]} {r.DQ_Score}" for r in dq_summary.itertuples()))
dq_summary

[Score] Trust DQ Score 96.8 | Outpatient 98.0, Inpatient 98.3, ED 92.4, Theatre 96.3, Workforce 98.8


,Dataset,Rows,Completeness_Pct,Validity_Pct,Uniqueness_Pct,Blank_Required_Cells,Rows_Failing_Rules,Duplicate_IDs,DQ_Score,RAG
0,Outpatient (PAS),580,99.78,94.14,100.00,10,34,0,98.0,Green
1,Inpatient (EHR),1206,98.40,97.10,99.50,174,35,6,98.3,Green
2,ED (EDIS),969,96.65,81.63,99.07,195,178,9,92.4,Amber
3,Theatre (TMS),208,95.75,93.27,100.00,53,14,0,96.3,Green
4,Workforce (WRS),1560,99.46,96.92,100.00,42,48,0,98.8,Green


## 4b. Record-level exceptions for the outpatient file
These are the rows Sarah previously found by eye. Each one has a reason and a suggested owner, so the list can be emailed straight to the PAS team.

In [9]:
exc = []
def add(mask, reason, owner):
    for _, r in op[mask].iterrows():
        exc.append({"Appointment_ID": r["Appointment_ID"], "Specialty_Code": r["Specialty_Code"],
                    "Field_Value": r.get("Attendance_Status"), "Exception": reason, "Owner": owner})
add(~op["Status_Valid"], "Non-standard Attendance_Status", "PAS team")
add(op["NHS_Blank"], "Blank NHS_Number - trace via PDS", "PAS team")
add(~op["Clinic_Valid"], "Retired / invalid Clinic_Code", "PAS team")
add(op["Waiting_Days_Flag"] == "NEGATIVE - excluded", "Negative Waiting_Days", "Patient Access")
add(op["Waiting_Days_Flag"] == "DERIVED from RTT_Weeks x7", "Waiting_Days held as text - derived", "Daniel Walsh (export mapping)")
add(op["Consultant_Code"] == "", "Blank Consultant_Code", "PAS team")
dq_exceptions = pd.DataFrame(exc)

wtr = (w[num(w["Hours_Worked"]) > WTR_MAX_HOURS]
       [["Shift_ID", "Staff_ID", "Ward_Code", "Shift_Date", "Shift_Type", "Hours_Rostered", "Hours_Worked"]])
log("Score", f"Outpatient record exceptions: {len(dq_exceptions)}; WTR >13h shifts: {len(wtr)}")
dq_exceptions["Exception"].value_counts()

[Score] Outpatient record exceptions: 41; WTR >13h shifts: 8


Exception
Waiting_Days held as text - derived    28
Blank Consultant_Code                   5
Blank NHS_Number - trace via PDS        3
Non-standard Attendance_Status          2
Retired / invalid Clinic_Code           2
Negative Waiting_Days                   1
Name: count, dtype: int64

## 5. Export: formatted Excel exception report
The workbook is written with **openpyxl** and has these sheets:

1. **Summary**: headline KPIs with RAG status, ready for the Operations Director
2. **DNA_by_Specialty**: with conditional fill against the 8% target
3. **RTT_by_Specialty**: with RAG against the 92% standard
4. **RTT_Breach_List**: patient-level list of open pathways over 18 weeks, longest first
5. **Waiting_Stats**: mean, median, P90 and max by specialty
6. **DQ_Scores**: score by dataset, plus a rule-level breakdown
7. **DQ_Exceptions**: record-level outpatient exceptions with owners
8. **WTR_Breaches**: shifts over the 13-hour limit
9. **Run_Log**: audit trail of every step in this run

In [10]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

NHS_BLUE, DARK_BLUE = "005EB8", "003087"
FILL = {"Green": "C6EFCE", "Amber": "FFEB9C", "Red": "FFC7CE"}
thin = Side(style="thin", color="BFBFBF")

summary = pd.DataFrame([
    ["Outpatient appointments in extract", len(op), "", ""],
    ["Appointments analysed (valid status)", len(op_clean), "", ""],
    ["DNA rate (DNA / ATT+DNA)", f"{trust_dna}%", "< 8%", "Green" if trust_dna < 8 else "Amber" if trust_dna <= 10 else "Red"],
    ["Specialties above 8% DNA target", int((dna["DNA_Rate_Pct"] >= DNA_TARGET_PCT).sum()), "0", ""],
    ["Open RTT pathways", rtt_total, "", ""],
    ["RTT breaches (open > 18 weeks)", int(rtt_breaches), "0", "Red" if rtt_breaches else "Green"],
    ["RTT 18-week compliance (open pathways)", f"{rtt_compliance}%", ">= 92%", "Green" if rtt_compliance >= 92 else "Amber" if rtt_compliance >= 85 else "Red"],
    ["Open pathways > 52 weeks", int(over_52), "0", "Red" if over_52 else "Green"],
    ["Open pathways > 104 weeks", int(over_104), "0", "Red" if over_104 else "Green"],
    ["Median wait, open pathways (days)", trust_wait["Median_Days"], "", ""],
    ["90th percentile wait, open pathways (days)", trust_wait["P90_Days"], "", ""],
    ["Trust DQ Score (average of 5 datasets)", trust_dq, ">= 95", "Green" if trust_dq >= 95 else "Amber" if trust_dq >= 85 else "Red"],
    ["Outpatient record-level DQ exceptions", len(dq_exceptions), "0", ""],
    ["Shifts over 13h (Working Time Regulations)", len(wtr), "0", "Red" if len(wtr) else "Green"],
], columns=["Measure", "Value", "Target", "RAG"])

out_file = OUT_DIR / f"Waiting_List_Exception_Report_{RUN_DATE:%Y-%m-%d}.xlsx"
sheets = {"Summary": summary, "DNA_by_Specialty": dna, "RTT_by_Specialty": rtt_spec,
          "RTT_Breach_List": breach_list, "Waiting_Stats": wait_stats,
          "DQ_Scores": dq_summary, "DQ_Exceptions": dq_exceptions, "WTR_Breaches": wtr,
          "Run_Log": pd.DataFrame(run_log)}

with pd.ExcelWriter(out_file, engine="openpyxl") as xw:
    for name, df in sheets.items():
        df.to_excel(xw, sheet_name=name, index=False, startrow=2)
    dq_rules.to_excel(xw, sheet_name="DQ_Scores", index=False, startrow=len(dq_summary) + 6)

wb = load_workbook(out_file)
titles = {"Summary": "Weekly Waiting List Exception Report",
          "DNA_by_Specialty": "DNA rate by specialty - DNA / (ATT + DNA), target < 8%",
          "RTT_by_Specialty": "RTT 18-week compliance by specialty - open pathways, standard 92%",
          "RTT_Breach_List": "Patient-level RTT breaches - open pathways over 18 weeks, longest first",
          "Waiting_Stats": "Waiting time statistics - open RTT pathways (days)",
          "DQ_Scores": "Data Quality Score by dataset (completeness, validity, uniqueness)",
          "DQ_Exceptions": "Outpatient record-level data quality exceptions",
          "WTR_Breaches": "Shifts over the 13-hour Working Time Regulations limit",
          "Run_Log": "Run log - audit trail"}
for ws in wb.worksheets:
    ws["A1"] = titles[ws.title]
    ws["A1"].font = Font(name="Arial", size=14, bold=True, color=DARK_BLUE)
    ws["A2"] = f"Bronings University Hospital Group | Generated {RUN_DATE:%d/%m/%Y %H:%M} | Source: PAS/EHR/EDIS/TMS/WRS extracts"
    ws["A2"].font = Font(name="Arial", size=9, italic=True, color="595959")
    header_rows = [3] + ([len(dq_summary) + 7] if ws.title == "DQ_Scores" else [])
    for row in ws.iter_rows(min_row=3):
        for c in row:
            if c.value is None: continue
            c.font = Font(name="Arial", size=10)
            c.border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for hr in header_rows:
        for c in ws[hr]:
            if c.value is not None:
                c.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
                c.fill = PatternFill("solid", fgColor=NHS_BLUE)
                c.alignment = Alignment(wrap_text=True, vertical="center")
    for col in ws.columns:
        width = max(len(str(c.value)) if c.value is not None and c.row > 2 else 0 for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(width + 2, 10), 50)
    ws.freeze_panes = "A4"
    # RAG colouring wherever a RAG / target column exists
    headers = {c.value: c.column for c in ws[3] if c.value}
    for key in ("RAG", "vs_8pct_Target"):
        if key in headers:
            for r in range(4, ws.max_row + 1):
                cell = ws.cell(row=r, column=headers[key])
                v = cell.value
                tag = {"Below target": "Green", "ABOVE TARGET": "Red"}.get(v, v)
                if tag in FILL:
                    cell.fill = PatternFill("solid", fgColor=FILL[tag]); cell.font = Font(name="Arial", size=10, bold=True)
wb["Summary"].column_dimensions["A"].width = 46
wb.save(out_file)

elapsed = round(time.time() - RUN_START, 1)
log("Export", f"Saved {out_file} in {elapsed} s")
summary

[Export] Saved output/Waiting_List_Exception_Report_2026-11-12.xlsx in 0.6 s


,Measure,Value,Target,RAG
0,Outpatient appointments in extract,580,,
1,Appointments analysed (valid status),578,,
2,DNA rate (DNA / ATT+DNA),14.1%,< 8%,Red
3,Specialties above 8% DNA target,6,0,
4,Open RTT pathways,130,,
5,RTT breaches (open > 18 weeks),76,0,Red
6,RTT 18-week compliance (open pathways),41.54%,>= 92%,Red
7,Open pathways > 52 weeks,2,0,Red
8,Open pathways > 104 weeks,0,0,Green
9,"Median wait, open pathways (days)",156.0,,


## 6. Scheduling (Monday 07:00) and handover notes for Daniel
**Run headless** (no Jupyter UI), from the project folder:
```
jupyter nbconvert --to notebook --execute --inplace Bronings_Waiting_List_Exception_Report.ipynb
```
**Windows Task Scheduler:** create a Basic Task, set the trigger to Weekly, Monday, 07:00, and the action to *Start a program* `jupyter` with the arguments above. Set *Start in* to the project folder.
**Linux / server cron:** `0 7 * * 1 cd /path/Bronings_Automation && jupyter nbconvert --to notebook --execute --inplace Bronings_Waiting_List_Exception_Report.ipynb`

**What to add next**
1. Read directly from Snowflake (`snowflake-connector-python`) instead of CSV exports, so there is no manual export step.
2. Email the workbook to the Operations Director and specialty managers automatically (Outlook/Graph API).
3. Week-on-week trend: append each run's Summary to a history table and chart the movement.
4. Add the ED 4-hour and theatre exception checks, using the same pattern.
5. Unit tests (pytest) for the DNA and RTT functions, so a code change cannot silently alter a statutory figure.